In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt


In [5]:
df=pd.read_csv("../data/pmu_data.csv")
df.head()

,timestamp,Va_mag,Va_ang,Vb_mag,Vb_ang,Vc_mag,Vc_ang,Ia_mag,Ia_ang,Ib_mag,Ib_ang,Ic_mag,Ic_ang,frequency,rocof
0,2026-02-10 14:07:15,132243.95,176.93,131467.20,56.92,132248.10,-63.08,247.44,-3.71,246.75,-123.82,246.94,116.08,59.9988,-0.0350
1,2026-02-10 14:07:15,132243.22,176.92,131468.32,56.92,132247.88,-63.09,247.46,-3.69,246.94,-123.89,246.76,116.10,59.9994,0.0631
2,2026-02-10 14:07:15,132241.92,176.92,131469.08,56.92,132247.48,-63.09,247.50,-3.74,246.95,-123.89,246.76,116.08,60.0001,0.0450
3,2026-02-10 14:07:15,132242.01,176.92,131468.25,56.91,132246.94,-63.09,247.53,-3.78,246.98,-123.90,246.85,116.11,59.9996,-0.0460
4,2026-02-10 14:07:15,132243.03,176.91,131467.20,56.91,132246.70,-63.09,247.25,-3.73,247.01,-123.91,246.96,116.19,59.9989,-0.0572


In [7]:
df.isnull().sum()

timestamp    0
Va_mag       0
Va_ang       0
Vb_mag       0
Vb_ang       0
Vc_mag       0
Vc_ang       0
Ia_mag       0
Ia_ang       0
Ib_mag       0
Ib_ang       0
Ic_mag       0
Ic_ang       0
frequency    0
rocof        0
dtype: int64

In [11]:
def add_features(df):
    df = df.copy()

    # Voltage magnitude differences
    df['Vab_mag_diff'] = df['Va_mag'] - df['Vb_mag']
    df['Vbc_mag_diff'] = df['Vb_mag'] - df['Vc_mag']
    df['Vac_mag_diff'] = df['Va_mag'] - df['Vc_mag']

   
    # Voltage angle differences
    # (wrapped to [-180, 180] for stability)
    
    def angle_diff(a, b):
        diff = a - b
        return (diff + 180) % 360 - 180

    df['Vab_ang_diff'] = angle_diff(df['Va_ang'], df['Vb_ang'])
    df['Vbc_ang_diff'] = angle_diff(df['Vb_ang'], df['Vc_ang'])
    df['Vac_ang_diff'] = angle_diff(df['Va_ang'], df['Vc_ang'])

    
    # Current magnitude differences
    
    df['Iab_mag_diff'] = df['Ia_mag'] - df['Ib_mag']
    df['Ibc_mag_diff'] = df['Ib_mag'] - df['Ic_mag']
    df['Iac_mag_diff'] = df['Ia_mag'] - df['Ic_mag']

    
    # Current angle differences
    
    df['Iab_ang_diff'] = angle_diff(df['Ia_ang'], df['Ib_ang'])
    df['Ibc_ang_diff'] = angle_diff(df['Ib_ang'], df['Ic_ang'])
    df['Iac_ang_diff'] = angle_diff(df['Ia_ang'], df['Ic_ang'])

    

    # Voltage imbalance indicator
    df['V_mag_std'] = df[['Va_mag','Vb_mag','Vc_mag']].std(axis=1)

    # Current imbalance indicator
    df['I_mag_std'] = df[['Ia_mag','Ib_mag','Ic_mag']].std(axis=1)

    # Frequency dynamics
    df['freq_dev'] = df['frequency'] - 60  # or 50 depending system

    return df

In [17]:
df = add_features(df)
df.shape

(1623, 30)

In [19]:
df=df.drop(columns='timestamp')

In [23]:
#X=df.drop(columns='attack').values
#y=df['attack'].values

In [25]:
# from sklearn.preprocessing import MinMaxScaler

# scaler = MinMaxScaler(feature_range=(-1, 1))
# X = scaler.fit_transform(X)

In [27]:
from sklearn.model_selection import train_test_split
# X_train, X_test, y_train, y_test = train_test_split(
#     X, y, test_size=0.30, random_state=42)